In [ ]:
!pip install pymupdf nltk tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 82.9 MB/s eta 0:00:00


In [ ]:
import fitz  # PyMuPDF
import re
import nltk
import json
from tqdm import tqdm
from collections import Counter

nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
from google.colab import files

uploaded = files.upload()
pdf_files = list(uploaded.keys())

print("Uploaded files:", pdf_files)

Saving Lecture02_Data Preprocessing&DataVisulaization.pdf to Lecture02_Data Preprocessing&DataVisulaization.pdf
Uploaded files: ['Lecture02_Data Preprocessing&DataVisulaization.pdf']


In [ ]:
def extract_pages(file_name):
    doc = fitz.open(stream=uploaded[file_name], filetype="pdf")

    pages = []
    for i, page in enumerate(doc):
        text = page.get_text()
        pages.append({
            "page_num": i + 1,
            "text": text
        })

    return pages

In [ ]:
def detect_repeated_lines(pages, threshold=0.6):
    lines_all = []

    for p in pages:
        lines = p["text"].split("\n")
        # ناخد أول وآخر 3 سطور (غالبًا header/footer)
        candidate_lines = lines[:3] + lines[-3:]
        lines_all.extend(candidate_lines)

    counter = Counter(lines_all)
    total_pages = len(pages)

    repeated = set()
    for line, count in counter.items():
        if count / total_pages >= threshold and len(line.strip()) > 3:
            repeated.add(line.strip())

    return repeated


def remove_headers_footers(pages):
    repeated_lines = detect_repeated_lines(pages)

    cleaned_pages = []
    for p in pages:
        lines = p["text"].split("\n")
        new_lines = []

        for line in lines:
            if line.strip() not in repeated_lines:
                new_lines.append(line)

        cleaned_pages.append({
            "page_num": p["page_num"],
            "text": "\n".join(new_lines)
        })

    return cleaned_pages

In [ ]:
def clean_text(text):

    text = re.sub(r'http\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'page\s*\d+', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\b\d+\b', ' ', text)
    text = re.sub(r'[^\x00-\x7F\u0600-\u06FF]+', ' ', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [ ]:
def chunk_text(text, chunk_size=300, overlap=50):
    words = text.split()
    chunks = []

    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk_words = words[start:end]

        chunk = " ".join(chunk_words)
        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [ ]:
def process_pdfs(pdf_files, chunk_size=300, overlap=50):
    all_chunks = []

    for file in pdf_files:
        print(f"Processing: {file}")

        pages = extract_pages(file)
        pages = remove_headers_footers(pages)

        for p in pages:
            clean = clean_text(p["text"])
            chunks = chunk_text(clean, chunk_size, overlap)

            for i, chunk in enumerate(chunks):
                if len(chunk.strip()) < 50:
                    continue

                all_chunks.append({
                    "text": chunk,
                    "source": file,
                    "page": p["page_num"],
                    "chunk_id": i
                })

    return all_chunks

In [ ]:
all_chunks = process_pdfs(pdf_files)

print("Total chunks:", len(all_chunks))

Processing: Lecture02_Data Preprocessing&DataVisulaization.pdf
Total chunks: 30


In [ ]:
with open("chunks.json", "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)

print("Saved to chunks.json")

Saved to chunks.json


In [ ]:
for i in range(3):
    print(all_chunks[i])

{'text': 'Data Preprocessing and Visualization Dr. Amr M. Nagy Faculty of computers and Artificial Intelligence Benha University', 'source': 'Lecture02_Data Preprocessing&DataVisulaization.pdf', 'page': 1, 'chunk_id': 0}
{'text': 'What & Why preprocess the data? Data cleaning Data integration Data transformation Data reduction PAAS Group Data visualization What? Why? Benefits Techniques Who uses it? Types of Graphs, Tools, Techniques in programming, Best resources Outline', 'source': 'Lecture02_Data Preprocessing&DataVisulaization.pdf', 'page': 2, 'chunk_id': 0}
{'text': 'Data Preprocessing can be defined as a process of converting raw data into a format that is understandable and usable for further analysis. It is an important step in the Data Preparation stage. It ensures that the outcome of the analysis is accurate, complete, and consistent. Data preprocessing refers to the cleaning, transforming and integrating of data to make it ready for analysis. Data Preprocessing', 'source': '

# Embeddings & Vector DB

In [ ]:
# Model: intfloat/multilingual-e5-large
# DB: ChromaDB
!pip install sentence-transformers chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 103.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.0 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentele

In [ ]:
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer
import torch

# ── 1. Check GPU ─────────────────────────────────────────
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

GPU available: True
Device: Tesla T4


In [ ]:
# ── 2. Load chunks  ───────────────────────────
with open("chunks.json", "r", encoding="utf-8") as f:
    all_chunks = json.load(f)

print(f" Loaded {len(all_chunks)} chunks")

 Loaded 30 chunks


In [ ]:
# ── 3. Load Model ────────────────────────────────
MODEL_NAME = "intfloat/multilingual-e5-large"
print(f" Loading model: {MODEL_NAME}")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(MODEL_NAME, device=device)

print(f" Model loaded on: {device}")

 Loading model: intfloat/multilingual-e5-large


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

 Model loaded on: cuda


In [ ]:
# ── 4. Generate Embeddings ───────────────────────────────
texts = ["passage: " + chunk["text"] for chunk in all_chunks]

print(" Generating embeddings...")
embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    device=device
)

embeddings = np.array(embeddings).astype("float32")
print(f" Embeddings shape: {embeddings.shape}")

 Generating embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

 Embeddings shape: (30, 1024)


In [ ]:
# ── 5. Setup ChromaDB ────────────────────────────────────
chroma_client = chromadb.PersistentClient(path="./chroma_db")

try:
    chroma_client.delete_collection("rag_collection")
except:
    pass

collection = chroma_client.create_collection(
    name="rag_collection",
    metadata={"hnsw:space": "cosine"}  # cosine similarity
)

print(" ChromaDB collection created")

 ChromaDB collection created


In [ ]:
def search_top_k(query, top_k=5):
    # 1. Encode query
    query_vec = model.encode(
        ["query: " + query],
        normalize_embeddings=True,
        device=device
    ).tolist()

    # 2. Query ChromaDB
    results = collection.query(
        query_embeddings=query_vec,
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    docs = results["documents"][0]
    metas = results["metadatas"][0]
    dists = results["distances"][0]

    # 3. Display results
    print(f"\n Query: {query}\n")

    output = []
    for i, (doc, meta, dist) in enumerate(zip(docs, metas, dists)):
        score = 1 - dist  # cosine distance → similarity

        print(f"[{i+1}] Score: {score:.4f} | Page: {meta['page']} | Source: {meta['source']}")
        print(f"     {doc[:120]}...\n")

        output.append({
            "text": doc,
            "score": score,
            "page": meta["page"],
            "source": meta["source"]
        })

    return output

In [ ]:
# ── 7. Quick Test ────────────────────────────────────────
def search(query, top_k=5):
    query_vec = model.encode(
        ["query: " + query],
        normalize_embeddings=True,
        device=device
    ).tolist()

    results = collection.query(
        query_embeddings = query_vec,
        n_results        = top_k,
        include          = ["documents", "metadatas", "distances"]
    )

    print(f"\n Query: {query}\n")
    for rank, (doc, meta, dist) in enumerate(zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    )):
        print(f"  [{rank+1}] Score: {1-dist:.4f} | "
              f"Page: {meta['page']} | "
              f"Source: {meta['source']}")
        print(f"       {doc[:120]}...")
        print()

ids = [f"id_{i}" for i in range(len(all_chunks))]
metadatas = [
    {"source": chunk["source"], "page": chunk["page"]}
    for chunk in all_chunks
]

collection.add(
    ids=ids,
    embeddings=embeddings.tolist(),
    documents=[chunk["text"] for chunk in all_chunks],
    metadatas=metadatas
)

print(f"added {collection.count()} ")
# query
search(" what is Data visualization?")

added 30 

 Query:  what is Data visualization?

  [1] Score: 0.9214 | Page: 24 | Source: Lecture02_Data Preprocessing&DataVisulaization.pdf
       Data visualization is the practice of translating information into a visual context, such as a map or graph, to make dat...

  [2] Score: 0.9136 | Page: 26 | Source: Lecture02_Data Preprocessing&DataVisulaization.pdf
       Data visualization is one of the steps of the data science process, which states that after data has been collected, pro...

  [3] Score: 0.8765 | Page: 27 | Source: Lecture02_Data Preprocessing&DataVisulaization.pdf
       Data visualization is important for almost every career. It can be used by teachers to display student test results, by ...

  [4] Score: 0.8682 | Page: 29 | Source: Lecture02_Data Preprocessing&DataVisulaization.pdf
       Showing change over time Showing a part-to-whole composition Depicting flows and processes Looking at how data is distri...

  [5] Score: 0.8660 | Page: 2 | Source: Lecture02_Data 

In [ ]:
# ──  Save model name  ────────────────────────
config = {
    "embedding_model": MODEL_NAME,
    "device": device,
    "chroma_path": "./chroma_db",
    "collection_name": "rag_collection",
    "prefix_query": "query: ",
    "prefix_passage": "passage: "
}

with open(" config.json", "w") as f:
    json.dump(config, f, indent=2)

print(" Saved config.json for llm ")

 Saved config.json for llm 


In [ ]:
!pip install sentence-transformers chromadb rank_bm25

# Filtering based on Score

In [ ]:
def filter_by_score(results, threshold=0.45):

    filtered = [r for r in results if r["score"] >= threshold]

    print(f"\n  Score Filtering:")
    print(f"    before  : {len(results)} ")
    print(f"    after  : {len(filtered)}   (threshold={threshold})")

    if not filtered:
        print("No relevant results found")

    return filtered

# Hybrid Search (Vector + BM25)

- keyword matching + semantic matching

In [ ]:
from rank_bm25 import BM25Okapi

tokenized_corpus = [chunk["text"].lower().split() for chunk in all_chunks]
bm25 = BM25Okapi(tokenized_corpus)

print(" BM25 index ready")


def hybrid_search(query, top_k=10, alpha=0.6):
    global model, collection, config

    # ── 1. Vector Search scores ──────────────────────────
    query_vec = model.encode(
        [config["prefix_query"] + query],
        normalize_embeddings=True,
        device=device
    ).tolist()

    vec_results = collection.query(
        query_embeddings=query_vec,
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    # dict: text → vector_score
    vec_scores = {}
    vec_meta   = {}
    for doc, meta, dist in zip(
        vec_results["documents"][0],
        vec_results["metadatas"][0],
        vec_results["distances"][0]
    ):
        vec_scores[doc] = round(1 - dist, 4)
        vec_meta[doc]   = meta

    # ── 2. BM25 scores ───────────────────────────────────
    tokenized_query = query.lower().split()
    bm25_raw        = bm25.get_scores(tokenized_query)

    # Normalize BM25 scores
    bm25_max = bm25_raw.max()
    if bm25_max > 0:
        bm25_norm = bm25_raw / bm25_max
    else:
        bm25_norm = bm25_raw

    # ── 3vector_score+BM25 score ────────────────────────────────
    combined = {}

    #  vector scores
    for doc, vscore in vec_scores.items():
        combined[doc] = {"vector": vscore, "bm25": 0.0, "meta": vec_meta[doc]}

    #  BM25 scores
    for i, chunk in enumerate(all_chunks):
        text = chunk["text"]
        if text in combined:
            combined[text]["bm25"] = round(float(bm25_norm[i]), 4)

    # ── 4. hybrid score ─────────────────────────
    output = []
    for doc, scores in combined.items():
        hybrid_score = alpha * scores["vector"] + (1 - alpha) * scores["bm25"]
        output.append({
            "text"        : doc,
            "score"       : round(hybrid_score, 4),
            "vector_score": scores["vector"],
            "bm25_score"  : scores["bm25"],
            "page"        : scores["meta"]["page"],
            "source"      : scores["meta"]["source"]
        })

    # ordering
    output = sorted(output, key=lambda x: x["score"], reverse=True)[:top_k]

    print(f"\n🔀 Hybrid Search (α={alpha}):")
    for i, r in enumerate(output[:3]):
        print(f"  [{i+1}] Hybrid: {r['score']:.4f} | "
              f"Vec: {r['vector_score']:.4f} | "
              f"BM25: {r['bm25_score']:.4f} | "
              f"Page: {r['page']}")

    return output

 BM25 index ready


# Reranking by cross-encoder

In [ ]:
from sentence_transformers import CrossEncoder

# ── Initialize Cross-Encoder Reranker ────────────────────────
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device=device)
print("Cross-Encoder reranker initialized.")

def rerank_results(query, results, top_n=5):
    if not results:
        print("No results to rerank")
        return []

    # Cross-Encoder  pairs: (query, chunk_text)
    pairs     = [(query, r["text"]) for r in results]
    ce_scores = reranker.predict(pairs)

    for i, r in enumerate(results):
        r["rerank_score"] = round(float(ce_scores[i]), 4)

    reranked = sorted(results, key=lambda x: x["rerank_score"], reverse=True)

    print(f"\n Reranking (Top {top_n}):")
    for i, r in enumerate(reranked[:top_n]):
        print(f"  [{i+1}] Rerank: {r['rerank_score']:.4f} | "
              f" before: {r['score']:.4f} | "
              f"Page: {r['page']}")
        print(f"       {r['text'][:100]}...")

    return reranked[:top_n]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-Encoder reranker initialized.


In [ ]:
def enhanced_search(query, top_k=10, threshold=0.45, final_top=3, use_hybrid=True, alpha=0.6):
    """
    [Vector Search or Hybrid Search]
            ↓
    [Score Filtering]
            ↓
    [Reranking]
    """
    print(f"\n{'='*55}")
    print(f"🔍 Query: {query}")
    print(f"{'='*55}")

    # Step 1: Search
    if use_hybrid:
        results = hybrid_search(query, top_k=top_k, alpha=alpha)
    else:
        results = vector_search(query, top_k=top_k)
        print(f"\n Vector Search: {len(results)} ")

    # Step 2: Filter
    results = filter_by_score(results, threshold=threshold)
    if not results:
        return []

    # Step 3: Rerank
    results = rerank_results(query, results, top_n=final_top)
    print(f" {len(results)} chunk ready for LLM")

    return results


# ── check ──────────────────────────────────────────────
results = enhanced_search(
    query      = "What is Data visualization?",
    top_k      = 10,       # taken from DB
    threshold  = 0.60,     # for score
    final_top  = 3,        # final result
    use_hybrid = True,     # True = Hybrid | False = Vector only
    alpha      = 0.6       # w for Hybrid (60 for semantic and 40 for Keywords)
)


🔍 Query: What is Data visualization?

🔀 Hybrid Search (α=0.6):
  [1] Hybrid: 0.9120 | Vec: 0.8534 | BM25: 1.0000 | Page: 2
  [2] Hybrid: 0.7131 | Vec: 0.9103 | BM25: 0.4173 | Page: 24
  [3] Hybrid: 0.7093 | Vec: 0.9010 | BM25: 0.4217 | Page: 26

  Score Filtering:
    before  : 10 
    after  : 6   (threshold=0.6)

 Reranking (Top 3):
  [1] Rerank: 11.3668 |  before: 0.7131 | Page: 24
       Data visualization is the practice of translating information into a visual context, such as a map o...
  [2] Rerank: 11.0885 |  before: 0.7093 | Page: 26
       Data visualization is one of the steps of the data science process, which states that after data has...
  [3] Rerank: 8.8596 |  before: 0.6509 | Page: 27
       Data visualization is important for almost every career. It can be used by teachers to display stude...
 3 chunk ready for LLM
